# 03 — Ghép dữ liệu ngoài

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = ROOT / "data" / "processed"
EXT = ROOT / "data" / "external"
OUT_DIR = ROOT / "results" / "data_quality"
TS = "Local Time"

ROOT, PROC, EXT

(WindowsPath('e:/Document/hoankiem-air-quality-forecasting'),
 WindowsPath('e:/Document/hoankiem-air-quality-forecasting/data/processed'),
 WindowsPath('e:/Document/hoankiem-air-quality-forecasting/data/external'))

## Bước 1 — Đọc dữ liệu gốc và bỏ cột không dùng

In [2]:
# AQI, UV Index, Solar Radiation giữ lại cho EDA; 05_features không dùng
DROP_BASE = ["UTC Time", "City", "Country Code", "Timezone"]

base = pd.read_csv(PROC / "cleaned_data.csv")
print("gốc:", base.shape)
base = base.drop(columns=[c for c in DROP_BASE if c in base.columns])
base[TS] = pd.to_datetime(base[TS])
base = base.set_index(TS).sort_index()
base.shape, list(base.columns)

gốc: (40608, 21)


((40608, 16),
 ['AQI',
  'CO',
  'NO2',
  'O3',
  'PM10',
  'PM25',
  'SO2',
  'Temperature',
  'Relative Humidity',
  'Precipitation',
  'Clouds',
  'Pressure',
  'UV Index',
  'Wind Speed',
  'Wind Direction',
  'Solar Radiation'])

## Bước 2 — Ghép ERA5-Land

In [3]:
era5land = pd.read_csv(EXT / "open_meteo" / "era5land_hoankiem_hourly.csv")
era5land["time"] = pd.to_datetime(era5land["time"])
era5land = era5land.set_index("time").sort_index()

ERA5LAND_KEEP = {
    "soil_moisture_0_to_7cm": "Soil Moisture",
    "boundary_layer_height": "Boundary Layer Height",
}
era5land_sel = era5land[list(ERA5LAND_KEEP)].rename(columns=ERA5LAND_KEEP)
print("bỏ (trùng dữ liệu gốc):", [c for c in era5land.columns if c not in ERA5LAND_KEEP])
era5land_sel.reindex(base.index).describe().round(3)

bỏ (trùng dữ liệu gốc): ['source', 'surface_pressure', 'temperature_2m', 'wind_speed_10m', 'wind_direction_10m', 'shortwave_radiation', 'precipitation']


,Soil Moisture,Boundary Layer Height
count,40608.000,36240.000
mean,0.314,445.206
std,0.087,371.479
min,0.056,10.000
25%,0.261,145.000
50%,0.328,340.000
75%,0.384,655.000
max,0.442,2615.000


In [4]:

base = base.join(era5land_sel.reindex(base.index), how="left")
base.shape

(40608, 18)

## Bước 3 — NASA POWER

In [5]:
power = pd.read_csv(EXT / "open_meteo" / "nasa_power_hoankiem_hourly.csv")
power["Local Time"] = pd.to_datetime(power["Local Time"])
power = power.set_index("Local Time").sort_index()

POWER_KEEP = {"T2MDEW": "Dew Point", "WS10M": "Wind Speed 10m"}
power_sel = power[list(POWER_KEEP)].rename(columns=POWER_KEEP)
print("bỏ (trùng dữ liệu gốc):", [c for c in power.columns if c not in POWER_KEEP and c not in ("key", "source", "year")])
power_sel.describe().round(2)

bỏ (trùng dữ liệu gốc): ['T2M', 'RH2M', 'WS2M', 'WD2M', 'PRECTOTCORR', 'PS', 'CLOUD_AMT']


,Dew Point,Wind Speed 10m
count,40608.00,40608.00
mean,20.58,2.65
std,5.86,1.35
min,-1.47,0.01
25%,16.85,1.74
50%,22.72,2.38
75%,25.41,3.32
max,28.63,17.20


In [6]:
base = base.join(power_sel.reindex(base.index), how="left")
base.shape

(40608, 20)

## Bước 4 — NASA FIRMS (điểm cháy)

In [7]:
FIRMS_DIR = EXT / "firms"
LOCAL_TZ = "Asia/Ho_Chi_Minh"
raw_fire_path = FIRMS_DIR / "hanoi_fires_2022_2026.csv"
days = pd.date_range(base.index.min().normalize(), base.index.max().normalize(), freq="D")

if raw_fire_path.exists():
    det = pd.read_csv(raw_fire_path)
    acq_utc = pd.to_datetime(
        det["acq_date"].astype(str) + " " + det["acq_time"].astype(str).str.zfill(4),
        format="%Y-%m-%d %H%M",
    ).dt.tz_localize("UTC")
    det["day"] = acq_utc.dt.tz_convert(LOCAL_TZ).dt.tz_localize(None).dt.normalize()

    near = det[det["dist_km"] <= 100]
    count_100 = near.groupby("day").size().reindex(days, fill_value=0)
    frp_100 = near.groupby("day")["frp"].sum().reindex(days, fill_value=0)
    fires_sel = pd.DataFrame({
        "Fire Count 100km 3d": count_100.rolling(3, min_periods=1).sum(),
        "Fire FRP 100km 7d": frp_100.rolling(7, min_periods=1).sum(),
        # ffill chỉ dùng quá khứ; ngày chưa từng thấy cháy thì đặt xa (999 km)
        "Fire Dist Min": det.groupby("day")["dist_km"].min().reindex(days).ffill().fillna(999.0),
    }, index=days)
    print("Tính lại từ", len(det), "điểm cháy, gom theo ngày giờ Việt Nam")
else:
    print("CẢNH BÁO: không có", raw_fire_path.name, "- dùng file theo ngày cũ (gom theo ngày UTC).")
    fires = pd.read_csv(FIRMS_DIR / "hanoi_fire_daily_features.csv", parse_dates=["date"])
    fires = fires.set_index("date").sort_index()
    FIRMS_KEEP = {
        "fire_count_100km_roll3d": "Fire Count 100km 3d",
        "fire_frp_100km_roll7d": "Fire FRP 100km 7d",
        "fire_dist_min_km": "Fire Dist Min",
    }
    fires_sel = fires[list(FIRMS_KEEP)].rename(columns=FIRMS_KEEP).reindex(days)

fires_sel.head(3)

Tính lại từ 66208 điểm cháy, gom theo ngày giờ Việt Nam


,Fire Count 100km 3d,Fire FRP 100km 7d,Fire Dist Min
2022-01-13,0.0,0.0,999.000000
2022-01-14,0.0,0.0,183.623636
2022-01-15,0.0,0.0,183.623636


In [8]:
# Số liệu của ngày D chỉ có khi ngày D kết thúc, nên mọi giờ trong ngày D dùng số liệu của ngày D-1
fires_prev_day = fires_sel.shift(1, freq="D")
fire_hourly = fires_prev_day.reindex(base.index.normalize())
fire_hourly.index = base.index
base = base.join(fire_hourly, how="left")

# Kiểm tra: giá trị lúc 00:00 và 23:00 cùng một ngày phải bằng giá trị của ngày hôm trước
check_day = base.index.normalize()[len(base) // 2]
assert np.isclose(base.loc[check_day, "Fire Count 100km 3d"],
                  fires_sel.loc[check_day - pd.Timedelta(days=1), "Fire Count 100km 3d"], equal_nan=True)
base.shape

(40608, 23)

## Bước 5 — Kiểm tra và lưu

In [9]:
report = {
    "rows": int(len(base)),
    "columns": int(base.shape[1]),
    "start": str(base.index.min()),
    "end": str(base.index.max()),
    "duplicate_timestamps": int(base.index.duplicated().sum()),
    "missing": {c: int(v) for c, v in base.isna().sum().items() if v > 0},
}
print(json.dumps(report, indent=2, ensure_ascii=False))

{
  "rows": 40608,
  "columns": 23,
  "start": "2022-01-13 00:00:00",
  "end": "2026-08-31 23:00:00",
  "duplicate_timestamps": 0,
  "missing": {
    "AQI": 18,
    "Boundary Layer Height": 4368,
    "Fire Count 100km 3d": 24,
    "Fire FRP 100km 7d": 24,
    "Fire Dist Min": 24
  }
}


In [10]:
print("Cột cuối cùng:")
for i, c in enumerate(base.columns, 1):
    print(f"  {i:>2}. {c}")

Cột cuối cùng:
   1. AQI
   2. CO
   3. NO2
   4. O3
   5. PM10
   6. PM25
   7. SO2
   8. Temperature
   9. Relative Humidity
  10. Precipitation
  11. Clouds
  12. Pressure
  13. UV Index
  14. Wind Speed
  15. Wind Direction
  16. Solar Radiation
  17. Soil Moisture
  18. Boundary Layer Height
  19. Dew Point
  20. Wind Speed 10m
  21. Fire Count 100km 3d
  22. Fire FRP 100km 7d
  23. Fire Dist Min


In [11]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
out = PROC / "merged_data.csv"
base.reset_index().to_csv(out, index=False, encoding="utf-8")
(OUT_DIR / "merge_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("->", out, base.shape)

-> e:\Document\hoankiem-air-quality-forecasting\data\processed\merged_data.csv (40608, 23)
